# The Boundary Lever

## 01 — Predictions

**Committed 2026-08-15, before the deciding computations are run or read.**

---

## THE CLAIM

> **The Cayley-Dickson doubling boundary is a chiral mirror,**
> 
> $$ e_{i+H}\, e_{j+H} \;=\; e_j\, e_i \qquad H = \dim/2 $$
> 
> **whose exactly two fixed points — $e_0$ and $e_{\dim/2}$ — are precisely the
> indices belonging to no zero-divisor plane, at every level from $\dim = 16$ upward.**

One sentence. Falsifiable in one computation.


## Provenance of each prediction — read this before scoring

⚠ **Honesty note, and it matters.** The `dim = 16` and `dim = 32` census, and the
mirror result at those levels, were computed on 2026-08-15 **before this paper
existed**. They are the **originating observations** — the thing that suggested the
claim. They are NOT tests of it, and `02_data` labels them as such. Scoring a
hypothesis against the data that generated it is worth nothing.

The predictions below are what the claim says about data **not yet examined**.

| # | prediction | status when committed |
|---|---|---|
| **P1** | orphans at `dim = 128` are exactly `[0, 64]` | **never computed** — the flagship falsifier |
| **P2** | mirror is order-reversing at `dim = 128`, on all `62 x 61 = 3782` pairs | never computed |
| **P3** | `upper x upper` escapes the lower half `0 / 4096` times at `dim = 128` | never computed |
| **P4** | ZD diagonal census at `dim = 64` is `4116`, split `588 / 2940 / 588` | enumerated inside the orphan scan but **never printed or read** |
| **P5** | nullity at `dim = 64`: non-crossing = `16`; crossing = `8` and `24` | same caveat as P4 |

P4 and P5 carry a weaker guarantee than P1-P3 and it is stated rather than hidden:
the planes were enumerated during the `dim = 64` orphan sweep, but only the orphan
list was ever printed. The census itself has not been looked at.


## Where P4 and P5 come from

Not free parameters — extrapolations of the measured `16 -> 32` step.

**P4.** At `dim = 32` the census was `84` lower, `420` crossing, `84` upper, total
`588 = 84 x 7`, with crossing `= 5 x 84`. Test F showed the lower half keeps its
parent's zero divisors *exactly*. So if `dim = 64` repeats the pattern with the T32
census (`588`) in the role the sedenion census (`84`) played:

```
lower = upper = 588        crossing = 5 x 588 = 2940        total = 7 x 588 = 4116
```

**P5.** At `dim = 32`, nullity separated location *perfectly*: non-crossing divisors
all had nullity `8 = dim/4`; crossing had `4 = dim/8` and `12 = 3 dim/8`, never `8`.
Carried to `dim = 64`: non-crossing `16`, crossing `8` and `24`.

⚠ **P4 and P5 are the risky ones and that is the point.** A pattern read off a
single step is exactly the kind of thing that breaks at the next one. If they fail
while P1 holds, the claim survives and the extrapolation dies — those are separate
results and will be reported separately.


## The falsifier

> **Compute the orphans at `dim = 128`. If they are not exactly `[0, 64]`, the claim
> is dead.**

No reinterpretation is available. The claim says *every level from 16 upward*, and
128 is a level.

### One boundary condition, stated in advance so it is not a rescue

`dim = 8` (the octonions) is expected to fail and is **excluded by the claim's own
wording** ("from `dim = 16` upward"), not by an exception carved after the fact.
Octonions are a **division algebra**: they have no zero divisors at all, so every
index is trivially an orphan and the question is vacuous. Zero divisors first appear
at the sedenions — which is exactly where the claim begins.

This is recorded here, before the results, so that `dim = 8` returning all eight
indices in `02_data` reads as a confirmation of scope and not as a patch.


In [1]:
import sys, os, math, itertools
sys.path.insert(0, os.path.abspath('/home/rendier/Projects/ThePlace'))
import numpy as np
from ValaQuenta.modules.box_kite.maths import cd_multiplication_table
np.set_printoptions(linewidth=120)


In [2]:
def cd(level):
    """Cayley-Dickson table and dimension at a doubling level."""
    return cd_multiplication_table(level)

def basis_ops(tab, dim):
    """P[k] = the (dim x dim) matrix of LEFT multiplication by e_k.

    Precomputing these once turns every L_a for a basis-pair divisor into a
    single matrix add, instead of a dim^2 Python loop per candidate. Without
    this the dim=128 sweep is hours; with it, seconds."""
    P = np.zeros((dim, dim, dim))
    for i in range(dim):
        for j in range(dim):
            s, k = tab[(i, j)]        # box_kite returns (SIGN, INDEX)
            P[i, k, j] = s
    return P

def diagonal_L(P, i, j, sgn):
    """L_a for a = (e_i + sgn*e_j)/sqrt(2)."""
    return (P[i] + sgn * P[j]) / math.sqrt(2.0)

def zd_census(level, tol=1e-10, verbose=False):
    """Every (i<j, sign) whose diagonal is a zero divisor, with its nullity."""
    tab, dim = cd(level); H = dim // 2
    P = basis_ops(tab, dim)
    rows = []
    for i in range(dim):
        for j in range(i + 1, dim):
            for sgn in (1.0, -1.0):
                s = np.linalg.svd(diagonal_L(P, i, j, sgn), compute_uv=False)
                n = int((s < tol).sum())
                if n:
                    loc = 'lower' if j < H else ('upper' if i >= H else 'cross')
                    rows.append({'i': i, 'j': j, 'sign': int(sgn), 'nullity': n, 'loc': loc})
    return dim, H, rows

def orphans_from(dim, rows):
    used = {r['i'] for r in rows} | {r['j'] for r in rows}
    return [k for k in range(dim) if k not in used]

def mirror_test(level):
    """Is e_(i+H) e_(j+H) the lower product with the order REVERSED?"""
    tab, dim = cd(level); H = dim // 2
    idx = [k for k in range(1, H)]
    rev = fwd = tot = 0
    for i in idx:
        for j in idx:
            if i == j: continue
            up = tab[(i + H, j + H)]
            if up == tab[(j, i)]: rev += 1
            if up == tab[(i, j)]: fwd += 1
            tot += 1
    esc = sum(1 for i in range(H) for j in range(H) if tab[(i + H, j + H)][1] >= H)
    return {'dim': dim, 'H': H, 'reversed': rev, 'preserved': fwd, 'pairs': tot,
            'upper_escapes_lower': esc, 'upper_pairs': H * H}


In [3]:
# The predictions, committed as data.
PREDICTIONS = {
    'P1': {'what': 'orphans at dim=128', 'expect': [0, 64]},
    'P2': {'what': 'mirror order-reversing at dim=128', 'expect': {'reversed': 3782, 'preserved': 0, 'pairs': 3782}},
    'P3': {'what': 'upper*upper escapes lower at dim=128', 'expect': 0},
    'P4': {'what': 'ZD census at dim=64', 'expect': {'lower': 588, 'cross': 2940, 'upper': 588, 'total': 4116}},
    'P5': {'what': 'nullity classes at dim=64', 'expect': {'non_crossing': [16], 'crossing': [8, 24]}},
}
import json, pathlib
pathlib.Path('predictions.json').write_text(json.dumps(PREDICTIONS, indent=2))
for k, v in PREDICTIONS.items():
    print(f"{k}  {v['what']:<40} expect {v['expect']}")


P1  orphans at dim=128                       expect [0, 64]
P2  mirror order-reversing at dim=128        expect {'reversed': 3782, 'preserved': 0, 'pairs': 3782}
P3  upper*upper escapes lower at dim=128     expect 0
P4  ZD census at dim=64                      expect {'lower': 588, 'cross': 2940, 'upper': 588, 'total': 4116}
P5  nullity classes at dim=64                expect {'non_crossing': [16], 'crossing': [8, 24]}
